#Imporve pronounciation

**pipeline:**

YouTube Noorani Qaida → download once → approved timestamps → FFmpeg → 28 local reference clips + metadata → child selects letter → play reference → child records → Wav2Vec2 → target-letter score / 100 → feedback.


In [1]:
!pip -q uninstall -y torchaudio
!pip -q install -U transformers librosa soundfile gradio "yt-dlp[default]"
!apt -qq update > /dev/null && apt -qq install -y ffmpeg > /dev/null

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.8/30.8 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.4/53.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubu

In [2]:
import os
import subprocess

deno_bin = os.path.expanduser("~/.deno/bin")

if not os.path.exists(os.path.join(deno_bin, "deno")):
    subprocess.run(
        "curl -fsSL https://deno.land/install.sh | sh -s -- -y",
        shell=True, capture_output=True, text=True
    )

os.environ["PATH"] = deno_bin + os.pathsep + os.environ.get("PATH", "")

check = subprocess.run(["deno", "--version"], capture_output=True, text=True)
if check.returncode == 0:
    print("Deno JS runtime ready:", check.stdout.splitlines()[0])
else:
    print("Error")

Deno JS runtime ready: deno 2.9.5 (stable, release, x86_64-unknown-linux-gnu)


In [3]:
import shutil
import numpy as np
import librosa
import torch
import gradio as gr
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2ForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#preparing YouTube Video as dataset

In [4]:
VIDEO_ID = "wO2DRVC-g9w"
YOUTUBE_URL = f"https://youtu.be/{VIDEO_ID}"

# Dataset structure used by the application
DATASET_DIR = "noorani_reference_dataset"
AUDIO_DIR = os.path.join(DATASET_DIR, "audio")
METADATA_FILE = os.path.join(DATASET_DIR, "metadata.csv")

# Temporary source audio used only during data preparation
FULL_AUDIO = "noorani_full.wav"

# Optional authentication file ONLY if YouTube blocks Colab.
# It is not a reference audio upload.
COOKIES_FILE = "cookies.txt"

os.makedirs(AUDIO_DIR, exist_ok=True)

## Optional: YouTube authentication

Only use `cookies.txt` if YouTube blocks the Colab IP with a bot/sign-in check.
This is **not** an audio file. The reference audio still comes entirely from YouTube.


In [5]:
# No manual Noorani audio upload is required.
# If you already have cookies.txt in the Colab working directory, it will be used automatically.
print(
    " No Noorani audio upload is required. "
    "The reference dataset is generated from the YouTube source."
)


 No Noorani audio upload is required. The reference dataset is generated from the YouTube source.


# Approved Letter Timestamps

In [6]:
letter_clips = {
    "ا": (34, 36), "ب": (36, 38), "ت": (38, 40), "ث": (40, 43),
    "ج": (43, 45), "ح": (45, 48), "خ": (48, 50), "د": (50, 53),
    "ذ": (53, 55), "ر": (55, 58), "ز": (58, 60), "س": (60, 63),
    "ش": (63, 66), "ص": (66, 69), "ض": (70, 72), "ط": (72, 75),
    "ظ": (75, 77), "ع": (77, 79), "غ": (80, 82), "ف": (83, 85),
    "ق": (85, 88), "ك": (88, 90), "ل": (90, 93), "م": (93, 96),
    "ن": (96, 98), "و": (98, 100), "ه": (101, 103), "ي": (105, 108)
}
assert len(letter_clips) == 28
print("28 approved timestamp ranges")

REFERENCE_DIR = "noorani_letters"
os.makedirs(REFERENCE_DIR, exist_ok=True)

reference_files = {
    "ا": "ا.ogg", "ب": "ب.ogg", "ت": "ت.ogg", "ث": "ث.ogg",
    "ج": "ج.ogg", "ح": "ح.ogg", "خ": "خ.ogg", "د": "د.ogg",
    "ذ": "ذ.ogg", "ر": "ر.ogg", "ز": "ز.ogg", "س": "س.ogg",
    "ش": "ش.ogg", "ص": "ص.ogg", "ض": "ض.ogg", "ط": "ط.ogg",
    "ظ": "ظ.ogg", "ع": "ع.ogg", "غ": "غ.ogg", "ف": "ف.ogg",
    "ق": "ق.ogg", "ك": "ك.ogg", "ل": "ل.ogg", "م": "م.ogg",
    "ن": "ن.ogg", "و": "و.ogg", "ه": "ه.ogg", "ي": "ي.ogg"
}
print(" Reference mapping:", len(reference_files))

28 approved timestamp ranges
 Reference mapping: 28


# Build the Noorani Reference Dataset

This section converts the YouTube source into the actual dataset used by the application.

Dataset output:

```text
noorani_reference_dataset/
├── audio/
│   ├── ا.ogg
│   ├── ب.ogg
│   ├── ...
│   └── ي.ogg
└── metadata.csv
```

`metadata.csv` stores the letter, local audio path, approved start/end timestamps, and source URL.

If all 28 clips already exist, the download/cutting step is skipped.


In [7]:
import pandas as pd

def reference_clips_missing():
    return [
        letter
        for letter in letter_clips
        if not os.path.exists(
            os.path.join(AUDIO_DIR, reference_files[letter])
        )
    ]


def _yt_dlp_attempt(client=None, use_cookies=False):
    cmd = [
        "yt-dlp",
        "-x",
        "--audio-format", "wav",
        "--audio-quality", "0",
        "-o", FULL_AUDIO,
        "--force-overwrites",
    ]

    if client:
        cmd += [
            "--extractor-args",
            f"youtube:player_client={client}"
        ]

    if use_cookies and os.path.exists(COOKIES_FILE):
        cmd += ["--cookies", COOKIES_FILE]

    cmd.append(YOUTUBE_URL)

    label = client or "default"
    if use_cookies:
        label += " + cookies"

    print(f"⬇Downloading Noorani audio ({label})...")

    return subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )


def download_full_audio():
    if os.path.exists(FULL_AUDIO):
        print(f"Cached source audio found: {FULL_AUDIO}")
        return

    print("⬇ Downloading the Noorani source from YouTube...")

    clients = [None, "android", "ios"]
    last_error = ""

    cookie_options = [False]
    if os.path.exists(COOKIES_FILE):
        cookie_options.append(True)

    for use_cookies in cookie_options:
        for client in clients:

            result = _yt_dlp_attempt(
                client=client,
                use_cookies=use_cookies
            )

            if (
                result.returncode == 0
                and os.path.exists(FULL_AUDIO)
            ):
                print("Full Noorani audio downloaded.")
                return

            last_error = result.stderr

    raise RuntimeError(
        "Could not download the YouTube source.\n\n"
        + last_error
    )


def build_reference_dataset():

    missing = reference_clips_missing()

    if not missing:
        print(" All 28 reference clips already exist.")
        print(" YouTube download and FFmpeg cutting skipped.")
        return

    print(f" Missing reference clips: {len(missing)}")
    print("Missing:", missing)

    download_full_audio()

    rows = []

    for letter, (start, end) in letter_clips.items():

        output = os.path.join(
            AUDIO_DIR,
            reference_files[letter]
        )

        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-i", FULL_AUDIO,
                "-ss", str(start),
                "-to", str(end),
                "-vn",
                "-c:a", "libvorbis",
                output
            ],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

        rows.append({
            "letter": letter,
            "audio_file": f"audio/{reference_files[letter]}",
            "start_seconds": start,
            "end_seconds": end,
            "source_url": YOUTUBE_URL
        })

        print(
            f" {letter}: "
            f"{start}s → {end}s"
        )


    metadata = pd.DataFrame(rows)

    metadata.to_csv(
        METADATA_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n Noorani Reference Dataset created")
    print(f" {DATASET_DIR}")
    print(f" 28 reference clips")
    print(f" {METADATA_FILE}")


build_reference_dataset()


 Missing reference clips: 28
Missing: ['ا', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'و', 'ه', 'ي']
⬇ Downloading the Noorani source from YouTube...
⬇Downloading Noorani audio (default)...
Full Noorani audio downloaded.
 ا: 34s → 36s
 ب: 36s → 38s
 ت: 38s → 40s
 ث: 40s → 43s
 ج: 43s → 45s
 ح: 45s → 48s
 خ: 48s → 50s
 د: 50s → 53s
 ذ: 53s → 55s
 ر: 55s → 58s
 ز: 58s → 60s
 س: 60s → 63s
 ش: 63s → 66s
 ص: 66s → 69s
 ض: 70s → 72s
 ط: 72s → 75s
 ظ: 75s → 77s
 ع: 77s → 79s
 غ: 80s → 82s
 ف: 83s → 85s
 ق: 85s → 88s
 ك: 88s → 90s
 ل: 90s → 93s
 م: 93s → 96s
 ن: 96s → 98s
 و: 98s → 100s
 ه: 101s → 103s
 ي: 105s → 108s

 Noorani Reference Dataset created
 noorani_reference_dataset
 28 reference clips
 noorani_reference_dataset/metadata.csv


#  Verify the Noorani Reference Dataset

The application will only use reference clips from this dataset.


In [8]:
# Create metadata.csv from existing 28 reference clips

metadata_rows = []

for letter, (start, end) in letter_clips.items():

    filename = reference_files[letter]
    audio_path = os.path.join(
        AUDIO_DIR,
        filename
    )

    if os.path.exists(audio_path):

        metadata_rows.append({
            "letter": letter,
            "audio_file": f"audio/{filename}",
            "start_seconds": start,
            "end_seconds": end,
            "source_url": YOUTUBE_URL
        })

metadata = pd.DataFrame(metadata_rows)

metadata.to_csv(
    METADATA_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(" metadata.csv created")
print(f"Entries: {len(metadata)}")

display(metadata)

 metadata.csv created
Entries: 28


,letter,audio_file,start_seconds,end_seconds,source_url
0,ا,audio/ا.ogg,34,36,https://youtu.be/wO2DRVC-g9w
1,ب,audio/ب.ogg,36,38,https://youtu.be/wO2DRVC-g9w
2,ت,audio/ت.ogg,38,40,https://youtu.be/wO2DRVC-g9w
3,ث,audio/ث.ogg,40,43,https://youtu.be/wO2DRVC-g9w
4,ج,audio/ج.ogg,43,45,https://youtu.be/wO2DRVC-g9w
5,ح,audio/ح.ogg,45,48,https://youtu.be/wO2DRVC-g9w
6,خ,audio/خ.ogg,48,50,https://youtu.be/wO2DRVC-g9w
7,د,audio/د.ogg,50,53,https://youtu.be/wO2DRVC-g9w
8,ذ,audio/ذ.ogg,53,55,https://youtu.be/wO2DRVC-g9w
9,ر,audio/ر.ogg,55,58,https://youtu.be/wO2DRVC-g9w


In [9]:
# Verify all 28 clips and the dataset manifest

missing = reference_clips_missing()

if os.path.exists(METADATA_FILE):
    metadata = pd.read_csv(METADATA_FILE)
else:
    metadata = pd.DataFrame()

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(" Dataset:", DATASET_DIR)
print(" Reference clips:", 28 - len(missing), "/ 28")
print("Metadata exists:", os.path.exists(METADATA_FILE))
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

for letter, filename in reference_files.items():
    path = os.path.join(AUDIO_DIR, filename)
    status = "Found" if os.path.exists(path) else "Not Found"
    print(f"{status} {letter} → {path}")

if missing:
    raise RuntimeError(
        f" Missing reference clips: {missing}"
    )

if metadata.empty or len(metadata) != 28:
    raise RuntimeError(
        " metadata.csv must contain all 28 letters."
    )

print("\n Noorani Reference Dataset is ready: 28/28")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Dataset: noorani_reference_dataset
 Reference clips: 28 / 28
Metadata exists: True
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Found ا → noorani_reference_dataset/audio/ا.ogg
Found ب → noorani_reference_dataset/audio/ب.ogg
Found ت → noorani_reference_dataset/audio/ت.ogg
Found ث → noorani_reference_dataset/audio/ث.ogg
Found ج → noorani_reference_dataset/audio/ج.ogg
Found ح → noorani_reference_dataset/audio/ح.ogg
Found خ → noorani_reference_dataset/audio/خ.ogg
Found د → noorani_reference_dataset/audio/د.ogg
Found ذ → noorani_reference_dataset/audio/ذ.ogg
Found ر → noorani_reference_dataset/audio/ر.ogg
Found ز → noorani_reference_dataset/audio/ز.ogg
Found س → noorani_reference_dataset/audio/س.ogg
Found ش → noorani_reference_dataset/audio/ش.ogg
Found ص → noorani_reference_dataset/audio/ص.ogg
Found ض → noorani_reference_dataset/audio/ض.ogg
Found ط → noorani_reference_dataset/audio/ط.ogg
Found ظ → noorani_reference_dataset/audio/ظ.ogg
Found ع → noorani_reference_dataset/audio/

In [10]:

display(
    metadata[
        ["letter", "audio_file", "start_seconds", "end_seconds"]
    ]
)


,letter,audio_file,start_seconds,end_seconds
0,ا,audio/ا.ogg,34,36
1,ب,audio/ب.ogg,36,38
2,ت,audio/ت.ogg,38,40
3,ث,audio/ث.ogg,40,43
4,ج,audio/ج.ogg,43,45
5,ح,audio/ح.ogg,45,48
6,خ,audio/خ.ogg,48,50
7,د,audio/د.ogg,50,53
8,ذ,audio/ذ.ogg,53,55
9,ر,audio/ر.ogg,55,58


In [11]:
def get_reference_audio(letter):
    """Return the local Noorani reference clip for the selected letter."""

    if letter not in reference_files:
        raise ValueError(
            f"Unknown Arabic letter: {letter}"
        )

    path = os.path.join(
        AUDIO_DIR,
        reference_files[letter]
    )

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Reference audio missing: {path}"
        )

    return path


print(
    " Reference audio for ب:",
    get_reference_audio("ب")
)


 Reference audio for ب: noorani_reference_dataset/audio/ب.ogg


#  Downloading Wav2Vec2 Model

In [13]:
label_to_letter = {
    "Ain":"ع", "Alif":"ا", "Ba":"ب", "Daad":"ض", "Dal":"د",
    "Dhaa":"ظ", "Dhal":"ذ", "Faa":"ف", "Ghain":"غ", "Ha":"ه",
    "Haa":"ح", "Jeem":"ج", "Kaf":"ك", "Kha":"خ", "Laam":"ل",
    "Meem":"م", "Noon":"ن", "Qaf":"ق", "Raa":"ر", "Saad":"ص",
    "Seen":"س", "Sheen":"ش", "Ta":"ت", "Taa":"ط", "Tha":"ث",
    "Unknown":"Unknown", "Waw":"و", "Yaa":"ي", "Zay":"ز"
}
arabic_letters = metadata["letter"].tolist()

assert len(arabic_letters) == 28

print("Arabic letters:", len(arabic_letters))
print(arabic_letters)

Arabic letters: 28
['ا', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'و', 'ه', 'ي']


In [14]:
MODEL_ID = "masumtechnonext/wav2vec2-arabic-letter-verifier"

processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)
model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_ID).to(device)
model.eval()


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.36k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=

In [15]:
#  DEBUG MODEL LABELS

print("MODEL:", MODEL_ID)

print("\n Number of classes:")
print(model.config.num_labels)

print("\n id2label:")
print(model.config.id2label)

print("\n label2id:")
print(model.config.label2id)

MODEL: masumtechnonext/wav2vec2-arabic-letter-verifier

 Number of classes:
29

 id2label:
{0: 'Ain', 1: 'Alif', 2: 'Ba', 3: 'Daad', 4: 'Dal', 5: 'Dhaa', 6: 'Dhal', 7: 'Faa', 8: 'Ghain', 9: 'Ha', 10: 'Haa', 11: 'Jeem', 12: 'Kaf', 13: 'Kha', 14: 'Laam', 15: 'Meem', 16: 'Noon', 17: 'Qaf', 18: 'Raa', 19: 'Saad', 20: 'Seen', 21: 'Sheen', 22: 'Ta', 23: 'Taa', 24: 'Tha', 25: 'Unknown', 26: 'Waw', 27: 'Yaa', 28: 'Zay'}

 label2id:
{'Ain': 0, 'Alif': 1, 'Ba': 2, 'Daad': 3, 'Dal': 4, 'Dhaa': 5, 'Dhal': 6, 'Faa': 7, 'Ghain': 8, 'Ha': 9, 'Haa': 10, 'Jeem': 11, 'Kaf': 12, 'Kha': 13, 'Laam': 14, 'Meem': 15, 'Noon': 16, 'Qaf': 17, 'Raa': 18, 'Saad': 19, 'Seen': 20, 'Sheen': 21, 'Ta': 22, 'Taa': 23, 'Tha': 24, 'Unknown': 25, 'Waw': 26, 'Yaa': 27, 'Zay': 28}


In [38]:
# MODEL EVALUATION

from huggingface_hub import hf_hub_download
import json

MODEL_ID = "masumtechnonext/wav2vec2-arabic-letter-verifier"

cm = pd.read_csv(
    hf_hub_download(MODEL_ID, "test_confusion_matrix.csv")
).iloc[:, 1:].to_numpy()

accuracy = np.trace(cm) / cm.sum()

tp = np.diag(cm)
precision = tp / cm.sum(0)
recall = tp / cm.sum(1)
f1 = 2 * precision * recall / (precision + recall)

macro_precision = np.nanmean(precision)
macro_recall = np.nanmean(recall)
macro_f1 = np.nanmean(f1)

with open(
    hf_hub_download(MODEL_ID, "calibration.json")
) as f:
    calibration = json.load(f)

confidence_threshold = calibration["confidence_threshold"]

print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Precision: {macro_precision*100:.2f}%")
print(f"Recall: {macro_recall*100:.2f}%")
print(f"F1: {macro_f1*100:.2f}%")
print(f"Threshold: {confidence_threshold*100:.0f}%")

Accuracy: 99.65%
Precision: 99.66%
Recall: 99.66%
F1: 99.65%
Threshold: 86%


We used the pretrained model and extracted its original evaluation metrics from the model files. No training or fine-tuning was performed on our dataset.
The model has 28 Arabic letters + Unknown Classes.

# Audio Preprocessing

for child voice

In [16]:
TARGET_SR = 16000

def preprocess_audio(audio, sample_rate=TARGET_SR):
    audio = np.asarray(audio, dtype=np.float32)

    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    audio, _ = librosa.effects.trim(audio, top_db=25)

    if len(audio) == 0:
        raise ValueError("لم يتم اكتشاف صوت.")

    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak

    if sample_rate != TARGET_SR:
        audio = librosa.resample(
            audio, orig_sr=sample_rate, target_sr=TARGET_SR
        )

    return audio

# predict_letter

In [17]:
def predict_letter(audio, target_letter):
    audio = preprocess_audio(audio, TARGET_SR)

    inputs = processor(
        audio, sampling_rate=TARGET_SR, return_tensors="pt"
    )

    with torch.no_grad():
        logits = model(inputs.input_values.to(device)).logits

    probabilities = torch.softmax(logits, dim=-1)[0]

    pred_id = int(torch.argmax(probabilities))
    predicted_label = model.config.id2label[pred_id]
    predicted_letter = label_to_letter.get(predicted_label, "Unknown")
    confidence = float(probabilities[pred_id]) * 100

    target_label = next(
        (label for label, letter in label_to_letter.items()
         if letter == target_letter), None
    )

    target_id = next(
        (int(idx) for idx, label in model.config.id2label.items()
         if label == target_label), None
    )

    target_probability = (
        float(probabilities[target_id]) * 100
        if target_id is not None else 0.0
    )

    is_correct = predicted_letter == target_letter
    score = round(target_probability, 2)

    if is_correct:
        if score >= 90:
            feedback = "ممتاز! "
        elif score >= 75:
            feedback = "أحسنت! "
        elif score >= 60:
            feedback = "جيد! حاول مرة أخرى لدرجة أعلى "
        else:
            feedback = "قريب! اسمع المثال وحاول مرة أخرى "
    elif predicted_letter == "Unknown":
        feedback = f"لم أتعرف على الحرف {target_letter}. اسمع المثال وحاول مرة أخرى."
    else:
        feedback = (
            f"النموذج سمع {predicted_letter}. المطلوب {target_letter}. "
            "اسمع المثال وحاول مرة أخرى."
        )

    k = min(5, len(probabilities))
    values, ids = torch.topk(probabilities, k=k)
    top5 = []

    for value, idx in zip(values, ids):
        label = model.config.id2label[int(idx)]
        top5.append({
            "letter": label_to_letter.get(label, "Unknown"),
            "label": label,
            "confidence": round(float(value) * 100, 2)
        })

    return {
        "target_letter": target_letter,
        "predicted_letter": predicted_letter,
        "predicted_label": predicted_label,
        "confidence": round(confidence, 2),
        "target_probability": round(target_probability, 2),
        "score": score,
        "is_correct": is_correct,
        "feedback": feedback,
        "top5": top5
    }

In [36]:
# CLASS for Improve Pronounciation


class PronunciationEvaluator:

    def __init__(self, threshold=86.0):
        self.threshold = threshold

    def evaluate(self, audio, target_letter):


        result = predict_letter(
            audio,
            target_letter
        )

        target_probability = result["target_probability"]

        if (
            result["predicted_letter"] == target_letter
            and target_probability >= self.threshold
        ):

            return {
                "success": True,
                "score": round(target_probability, 2),
                "message": (
                    f"أتقنت الحرف بنسبة "
                    f"{target_probability:.0f}% "
                )
            }


        return {
            "success": False,
            "score": None,
            "message": "عيد التسجيل وحاول مرة ثانية "
        }



pronunciation_evaluator = PronunciationEvaluator(
    threshold=86.0
)

print(" Acceptance threshold:", pronunciation_evaluator.threshold)


 Acceptance threshold: 86.0


In [44]:
# TEST ALL 28 LETTERS

from google.colab import files
import librosa

uploaded = files.upload()
results = []

for filename in uploaded.keys():

    letter = next(
        (x for x in filename if x in arabic_letters or x == "أ"),
        None
    )

    letter = "ا" if letter == "أ" else letter

    if letter not in arabic_letters:
        print(f"Skipped: {filename}")
        continue

    audio, sr = librosa.load(
        filename,
        sr=16000,
        mono=True
    )

    result = predict_letter(
        audio,
        target_letter=letter
    )

    confidence = result["confidence"]
    prediction = result["predicted_letter"]

    accepted = (
        prediction == letter
        and confidence >= 86
    )

    results.append(accepted)

    print(
        f"{letter}: {prediction} | "
        f"{confidence:.2f}% | "
        f"{'قبول التسجيل' if accepted else 'اعد التسجيل'}"
    )

print(f"\nAccepted: {sum(results)}/{len(results)}")
print(f"Accuracy: {sum(results)/len(results)*100:.2f}%")

Saving أ.ogg to أ (4).ogg
Saving ب.ogg to ب (3).ogg
Saving ت.ogg to ت (3).ogg
Saving ث.ogg to ث (3).ogg
Saving ج.ogg to ج (3).ogg
Saving ح.ogg to ح (3).ogg
Saving خ.ogg to خ (3).ogg
Saving د.ogg to د (3).ogg
Saving ذ.ogg to ذ (3).ogg
Saving ر.ogg to ر (3).ogg
Saving ز.ogg to ز (3).ogg
Saving س.ogg to س (3).ogg
Saving ش.ogg to ش (3).ogg
Saving ص.ogg to ص (3).ogg
Saving ض.ogg to ض (3).ogg
Saving ط.ogg to ط (3).ogg
Saving ظ.ogg to ظ (3).ogg
Saving ع.ogg to ع (3).ogg
Saving غ.ogg to غ (3).ogg
Saving ف.ogg to ف (3).ogg
Saving ق.ogg to ق (3).ogg
Saving ك.ogg to ك (3).ogg
Saving ل.ogg to ل (3).ogg
Saving م.ogg to م (3).ogg
Saving ن.ogg to ن (3).ogg
Saving ه.ogg to ه (3).ogg
Saving و.ogg to و (3).ogg
Saving ي.ogg to ي (3).ogg
ا: ا | 90.72% | قبول التسجيل
ب: Unknown | 38.83% | اعد التسجيل
ت: ت | 89.03% | قبول التسجيل
ث: ث | 88.91% | قبول التسجيل
ج: ج | 83.12% | اعد التسجيل
ح: ح | 83.69% | اعد التسجيل
خ: خ | 90.49% | قبول التسجيل
د: ز | 79.01% | اعد التسجيل
ذ: ذ | 90.93% | قبول التسجيل
ر: Unknow

The model was tested using one recording for each of the 28 Arabic letters. The test achieved 21/28 accepted pronunciations with 75% accuracy using an 86% confidence threshold. The 86% threshold is the calibrated threshold provided by the pretrained model itself. Lowering the threshold may improve recognition of valid pronunciations, but may also increase incorrect predictions. This practical test was conducted using our own recordings and is separate from the model's original test-set evaluation.

#  Testing

In [20]:
def test_audio_file(filename, target_letter):
    audio, sr = librosa.load(filename, sr=TARGET_SR, mono=True)
    result = predict_letter(audio, target_letter)

    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(" الحرف الحقيقي:", target_letter)
    print("التوقع:", result["predicted_letter"])
    print(f" Confidence: {result['confidence']:.2f}%")
    print(f" Target Probability: {result['target_probability']:.2f}%")
    print(f" Score: {result['score']:.2f}/100")
    print("", " صحيح" if result["is_correct"] else " خطأ")
    print("", result["feedback"])

    for i, item in enumerate(result["top5"], 1):
        print(f"{i}. {item['letter']} ({item['label']}) → {item['confidence']:.2f}%")

    return result


# Simple gardio to show the pipline


In [21]:
import gradio as gr
import numpy as np
import librosa

In [22]:
# ANALYZE CHILD VOICE


def analyze_child_voice(audio_path, target_letter):

    if audio_path is None:
        return (
            " سجل صوتك أولًا",
            0,
            "حاول تسجيل الحرف مرة أخرى."
        )

    try:


        audio, sr = librosa.load(
            audio_path,
            sr=16000,
            mono=True
        )

        print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
        print(" Microphone Debug")
        print(" File:", audio_path)
        print(" Sample Rate:", sr)
        print(
            "⏱Duration:",
            round(len(audio) / sr, 2),
            "seconds"
        )
        print(
            " Max amplitude:",
            round(
                float(np.max(np.abs(audio))),
                4
            )
        )
        print(
            " RMS:",
            round(
                float(np.sqrt(np.mean(audio ** 2))),
                4
            )
        )
        print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")


        # CLASS CALL HERE


        result = pronunciation_evaluator.evaluate(
            audio,
            target_letter
        )

        if result["success"]:

            return (
                " أحسنت! نطقت الحرف بشكل صحيح ",
                result["score"],
                result["message"]
            )

        else:

            return (
                " عيد التسجيل",
                0,
                result["message"]
            )

    except Exception as e:

        print(" ERROR:", repr(e))

        return (
            " حدث خطأ أثناء تحليل الصوت",
            0,
            str(e)
        )


In [23]:
#  Analyze Child Recording


def analyze_child(audio_path, target_letter):

    if audio_path is None:
        return (
            " سجل صوتك أولًا",
            0,
            "حاول تسجيل الحرف مرة أخرى.",
            ""
        )

    try:

        # Read microphone recording as WAV
        audio, sr = librosa.load(
            audio_path,
            sr=16000,
            mono=True
        )

        print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
        print(" Microphone Debug")
        print("File:", audio_path)
        print("Sample Rate:", sr)
        print(
            "Duration:",
            round(len(audio) / sr, 2),
            "seconds"
        )
        print(
            "Max amplitude:",
            round(float(np.max(np.abs(audio))), 4)
        )
        print(
            "RMS:",
            round(
                float(np.sqrt(np.mean(audio ** 2))),
                4
            )
        )
        print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")


        result = predict_letter(
            audio,
            target_letter
        )

        status = (
            f"صحيح — {result['predicted_letter']}"
            if result["is_correct"]
            else f"النموذج سمع: {result['predicted_letter']}"
        )

        top5 = "\n".join(
            f"{i}. "
            f"{item['label']} "
            f"({item['letter']}) → "
            f"{item['confidence']:.2f}%"
            for i, item in enumerate(
                result["top5"],
                1
            )
        )

        return (
            status,
            result["score"],
            result["feedback"],
            top5
        )

    except Exception as e:

        print(" ERROR:", repr(e))

        return (
            " حدث خطأ أثناء تحليل الصوت",
            0,
            str(e),
            ""
        )

In [24]:
# Load Noorani Dataset + Arabic Letters


import os
import pandas as pd

# Dataset paths
DATASET_DIR = "noorani_reference_dataset"
AUDIO_DIR = os.path.join(DATASET_DIR, "audio")
METADATA_FILE = os.path.join(DATASET_DIR, "metadata.csv")

# Check metadata
if not os.path.exists(METADATA_FILE):
    raise FileNotFoundError(
        f" metadata.csv not found:\n{METADATA_FILE}\n"
        "Run the Noorani Dataset preparation/metadata cell first."
    )

# Load metadata
metadata = pd.read_csv(
    METADATA_FILE,
    encoding="utf-8-sig"
)

# Get the 28 letters directly from the dataset
arabic_letters = metadata["letter"].tolist()

print( METADATA_FILE)
print( arabic_letters)
print( len(arabic_letters))

noorani_reference_dataset/metadata.csv
['ا', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'و', 'ه', 'ي']
28


In [39]:

# Simple GRADIO UI


def show_letter(letter):

    reference_path = get_reference_audio(letter)

    return (
        f" الحرف المختار: {letter}",
        reference_path
    )

print("✅ show_letter ready")

with gr.Blocks(title="تعلم الحروف العربية") as demo:

    gr.Markdown(
        "#  تعلم الحروف العربية\n"
        "### اختر حرفًا → اسمع → قلد → احصل على تقييمك "
    )

    with gr.Row():

        with gr.Column():


            letter = gr.Dropdown(
                choices=arabic_letters,
                value="ا",
                label=" اختر الحرف"
            )

            letter_display = gr.Textbox(
                value=" الحرف المختار: ا",
                label="الحرف",
                interactive=False
            )


            reference_audio = gr.Audio(
                value=get_reference_audio("ا"),
                type="filepath",
                label="نطق القاعدة النورانية",
                interactive=False
            )


            child_audio = gr.Audio(
                sources=["microphone"],
                type="filepath",
                format="wav",
                label=" سجل نطقك"
            )

            analyze_button = gr.Button(
                "حلّل نطقي",
                variant="primary"
            )

    gr.Markdown("---")
    gr.Markdown("##  تقييم النطق")

    with gr.Row():

        result_status = gr.Textbox(
            label=" النتيجة"
        )

        score_output = gr.Number(
            label=" Score / 100"
        )

    feedback_output = gr.Textbox(
        label="💡 Feedback"
    )



    letter.change(
        fn=show_letter,
        inputs=letter,
        outputs=[
            letter_display,
            reference_audio
        ]
    )



    analyze_button.click(
        fn=analyze_child_voice,
        inputs=[
            child_audio,
            letter
        ],
        outputs=[
            result_status,
            score_output,
            feedback_output
        ]
    )



✅ show_letter ready


In [40]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://93389a2a317072a4fe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Microphone Debug
 File: /tmp/gradio/959eb2c704d7c7c439ac38c130c89f8832ba27c4e4d0941e94cb37d6c3546197/audio.wav
 Sample Rate: 16000
⏱Duration: 2.28 seconds
 Max amplitude: 1.0453
 RMS: 0.123
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Microphone Debug
 File: /tmp/gradio/4b9c96deeef3d7d5b393e6897c3d20d03676a70b13a354484ce8b8e4da963bc7/audio.wav
 Sample Rate: 16000
⏱Duration: 2.16 seconds
 Max amplitude: 0.9722
 RMS: 0.1346
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Microphone Debug
 File: /tmp/gradio/4b9c96deeef3d7d5b393e6897c3d20d03676a70b13a354484ce8b8e4da963bc7/audio.wav
 Sample Rate: 16000
⏱Duration: 2.16 seconds
 Max amplitude: 0.9722
 RMS: 0.1346
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Microphone Debug
 File: /tmp/gradio/0e05752f9406c6026fabe0853431a1341c0ee7aba5ac5a2d7e786d48165ff643/audio.wav
 Sample Rate: 16000
⏱Duration: 1.92 seconds
 Max amplitude: 1.0016
 RMS: 0.1516
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
━